In [ ]:
"""
NOTEBOOK: HYPERPARAMETER TUNING
================================
Purpose: Optimize model parameters for best performance
Output: Best hyperparameters for production
"""

# 🔧 Hyperparameter Tuning Notebook

**Objective:** Find optimal parameters for Random Forest models

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

class HyperparameterTuner:
    """Optimize model hyperparameters"""
    
    def __init__(self, model_type='random_forest'):
        self.model_type = model_type
        self.best_params = {}
        self.best_score = 0
    
    def tune_random_forest(self, X, y):
        """Tune Random Forest hyperparameters"""
        print("🔧 Tuning Random Forest hyperparameters...")
        
        # Define parameter grid
        param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 15, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None]
        }
        
        # Create base model
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
        
        # Grid search
        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=param_grid,
            cv=5,
            scoring='f1',
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(X, y)
        
        self.best_params = grid_search.best_params_
        self.best_score = grid_search.best_score_
        
        print(f"\n✅ Best Parameters: {self.best_params}")
        print(f"✅ Best F1 Score: {self.best_score:.3f}")
        
        return grid_search.best_estimator_
    
    def visualize_tuning_results(self, cv_results):
        """Visualize hyperparameter tuning results"""
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # n_estimators effect
        n_estimators_results = []
        for params, score in zip(cv_results['params'], cv_results['mean_test_score']):
            if 'n_estimators' in params:
                n_estimators_results.append((params['n_estimators'], score))
        
        n_estimators_df = pd.DataFrame(n_estimators_results, columns=['n_estimators', 'score'])
        n_estimators_df.groupby('n_estimators')['score'].mean().plot(kind='bar', ax=axes[0,0])
        axes[0,0].set_title('Effect of n_estimators')
        
        # max_depth effect
        depth_results = []
        for params, score in zip(cv_results['params'], cv_results['mean_test_score']):
            if 'max_depth' in params:
                depth_results.append((params['max_depth'], score))
        
        depth_df = pd.DataFrame(depth_results, columns=['max_depth', 'score'])
        depth_df.groupby('max_depth')['score'].mean().plot(kind='bar', ax=axes[0,1])
        axes[0,1].set_title('Effect of max_depth')
        
        plt.tight_layout()
        plt.savefig('../../proofs/hyperparameter_tuning.png', dpi=150)
        plt.show()
    
    def save_best_params(self, version="v1"):
        """Save best hyperparameters to production"""
        import json
        import os
        
        os.makedirs("../../exports", exist_ok=True)
        
        params_path = f"../../exports/best_params_{version}.json"
        with open(params_path, 'w') as f:
            json.dump({
                'model_type': self.model_type,
                'best_params': self.best_params,
                'best_score': self.best_score,
                'timestamp': str(pd.Timestamp.now())
            }, f, indent=2)
        
        print(f"💾 Best parameters saved to: {params_path}")
        return params_path

print("\n✅ Hyperparameter tuning loaded!")